# Nearest-TSS Gene Evaluation — 26.03 Training Set

Assesses what fraction of the gold-standard positives in the final training
set (`2603_training_set_full_fm.parquet`) are the **closest gene to the
sentinel variant TSS** in their credible set, and whether eQTL colocalisation
scores differ between nearest and non-nearest TSS positives.

`distanceSentinelTssNeighbourhood == 1` → gene is the nearest TSS in its
credible-set neighbourhood; `0` → not the nearest.

In [1]:
import pyspark.sql.functions as f

In [2]:
"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )
    if Path(app_default_credentials).exists():
        return app_default_credentials
    raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize": "2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)


session = GentropySession()

Loading BokehJS ...

In [3]:
fm = session.spark.read.parquet("2603_training_set_full_fm.parquet").cache()
positives = fm.filter(f.col("GSP") == 1).cache()
print(f"Total training set rows:  {fm.count():>10,}")
print(f"Total positive rows:      {positives.count():>10,}")

26/05/08 21:13:34 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@74b6c06a{/SQL,null,AVAILABLE,@Spark}
26/05/08 21:13:34 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@56e0308a{/SQL/json,null,AVAILABLE,@Spark}
26/05/08 21:13:34 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@7f0cb126{/SQL/execution,null,AVAILABLE,@Spark}
26/05/08 21:13:34 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@29bce601{/SQL/execution/json,null,AVAILABLE,@Spark}
26/05/08 21:13:34 INFO ContextHandler: Started o.s.j.s.ServletContextHandler@682b38f2{/static/sql,null,AVAILABLE,@Spark}


Total training set rows:     200,536
Total positive rows:          13,060


## 1. Row-Level Evaluation

Each row is one (studyLocusId, geneId) pair. The question is: among all
positive rows, how many have the gene as the nearest TSS gene in that
credible set?

In [4]:
n_pos = positives.count()
n_nearest = positives.filter(f.col("distanceSentinelTssNeighbourhood") == 1).count()
n_not_nearest = n_pos - n_nearest

print(f"Positive rows total:          {n_pos:>8,}")
print(f"  nearest TSS  (feature = 1): {n_nearest:>8,}  ({n_nearest / n_pos * 100:.1f}%)")
print(f"  not nearest  (feature = 0): {n_not_nearest:>8,}  ({n_not_nearest / n_pos * 100:.1f}%)")

Positive rows total:            13,060
  nearest TSS  (feature = 1):    7,201  (55.1%)
  not nearest  (feature = 0):    5,859  (44.9%)


## 2. eQTL Colocalisation: Nearest vs Non-Nearest TSS

Do positives where the gold-standard gene is the nearest TSS show stronger
eQTL colocalisation than those where it is not? We compare `eQtlColocClppMaximum`
(max CLPP across all eQTL datasets in the credible-set neighbourhood) between
the two groups.

Nulls (no eQTL data) are treated as 0 for the distribution stats.

In [5]:
nearest     = positives.filter(f.col("distanceSentinelTssNeighbourhood") == 1)
not_nearest = positives.filter(f.col("distanceSentinelTssNeighbourhood") == 0)

# Replace nulls with 0 for both groups
col = "eQtlColocClppMaximum"
near_vals = nearest.withColumn(col, f.coalesce(f.col(col), f.lit(0.0))).select(col)
far_vals  = not_nearest.withColumn(col, f.coalesce(f.col(col), f.lit(0.0))).select(col)

In [6]:
# Percentile distribution for both groups
quantiles = [0.0, 0.25, 0.5, 0.75, 0.9, 0.95, 1.0]

near_q = near_vals.approxQuantile(col, quantiles, 0.001)
far_q  = far_vals.approxQuantile(col, quantiles, 0.001)

labels = ["min", "p25", "p50", "p75", "p90", "p95", "max"]
print(f"{'Percentile':<10} {'Nearest TSS':>14} {'Not Nearest':>14}")
print("-" * 40)
for lbl, nq, fq in zip(labels, near_q, far_q):
    print(f"{lbl:<10} {nq:>14.4f} {fq:>14.4f}")

Percentile    Nearest TSS    Not Nearest
----------------------------------------
min                0.0000         0.0000
p25                0.0000         0.0000
p50                0.0000         0.0000
p75                0.0586         0.0000
p90                0.2335         0.0167
p95                0.4321         0.0204
max                0.9500         0.0682


In [7]:
# Mean and fraction with any eQTL coloc signal (score > 0)
from pyspark.sql.functions import mean as spark_mean, sum as spark_sum, count as spark_count

def stats(df):
    return df.agg(
        spark_mean(col).alias("mean"),
        (spark_sum(f.when(f.col(col) > 0, 1).otherwise(0)) / spark_count("*")).alias("frac_nonzero"),
        spark_count("*").alias("n"),
    ).collect()[0]

sn = stats(near_vals)
sf = stats(far_vals)

print(f"{'':25} {'Nearest TSS':>14} {'Not Nearest':>14}")
print("-" * 55)
print(f"{'n':25} {sn.n:>14,} {sf.n:>14,}")
print(f"{'mean':25} {sn.mean:>14.4f} {sf.mean:>14.4f}")
print(f"{'fraction with score > 0':25} {sn.frac_nonzero:>14.3f} {sf.frac_nonzero:>14.3f}")

                             Nearest TSS    Not Nearest
-------------------------------------------------------
n                                  7,201            204
mean                              0.0779         0.0027
fraction with score > 0            0.372          0.127
